# Customer Segmentation using RFM Analysis and K-Means Clustering

## Project Objective
The goal of this project is to segment customers based on their purchasing behavior using **RFM analysis (Recency, Frequency, Monetary)** and **K-Means clustering**.

Customer segmentation helps businesses:
- Identify high-value customers
- Understand customer purchasing patterns
- Design targeted marketing strategies
- Improve customer retention

# Step 1: Import Required Libraries

We import libraries used for:

- Data manipulation → Pandas, NumPy
- Data visualization → Matplotlib, Seaborn
- Machine learning → Scikit-learn

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## Step 2: Load Dataset

We load the **Online Retail dataset**, which contains transaction data for a UK-based online store.

Key columns include:

- InvoiceNo → Transaction ID
- Quantity → Number of products purchased
- InvoiceDate → Date of purchase
- UnitPrice → Price per item
- CustomerID → Unique customer identifier

In [ ]:
df = pd.read_excel("../data/Online Retail.xlsx")

df.head()

## Step 3: Data Exploration

Before cleaning the data, we explore the dataset to understand:

- Number of rows and columns
- Data types
- Missing values

In [ ]:
df.shape

In [ ]:
df.info()

## Step 4: Data Cleaning

We clean the dataset by:

1. Removing rows where **CustomerID is missing**
2. Removing transactions with **negative quantity (returns)**
3. Converting **InvoiceDate to datetime**
4. Creating a new column **TotalPrice = Quantity × UnitPrice**

In [ ]:
df = df.dropna(subset=['CustomerID'])

In [ ]:
df = df[df['Quantity'] > 0]

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [ ]:
df.head()

## Step 5: RFM Feature Engineering

RFM stands for:

Recency → How recently a customer made a purchase  
Frequency → How often a customer purchases  
Monetary → Total amount spent by the customer

These metrics help measure **customer value and engagement**.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

In [ ]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [ ]:
rfm.head()

## Step 6: Data Scaling

Machine learning algorithms perform better when features are scaled.

We use **StandardScaler** to normalize Recency, Frequency, and Monetary values.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

## Step 7: Determine Optimal Number of Clusters

We use the **Elbow Method** to determine the optimal number of clusters.

The elbow point indicates the best number of clusters.

In [ ]:
inertia = []

for i in range(1,10):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

Plot the elbow graph

In [ ]:
plt.plot(range(1,10), inertia)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

## Step 8: Apply K-Means Clustering

Based on the elbow plot, we choose **4 clusters** and apply K-Means clustering to group customers.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)

rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
rfm.head()

## Step 9: Analyze Customer Clusters

We analyze the average Recency, Frequency, and Monetary values for each cluster.

In [ ]:
rfm.groupby('Cluster').mean()

Analyze Customer segments 

In [ ]:
rfm.groupby('Cluster').mean()

## Step 10: Visualize Customer Segments

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=rfm,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2'
)

plt.title("Customer Segmentation")
plt.show()

## Step 11: Create RFM Scores

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

## Step 12: Combine RFM Scores

In [ ]:
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

rfm.head()

## Step 13: Customer Segmentation

In [ ]:
def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4:
        return "Champions"

    elif row['F_Score'] >= 4:
        return "Loyal Customers"

    elif row['R_Score'] >= 4:
        return "New Customers"

    elif row['R_Score'] <= 2:
        return "At Risk"

    else:
        return "Others"

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

## Step 14: Segment Distribution

In [ ]:
rfm['Segment'].value_counts()

## Step 15: Customer Segment Visualizations

In [ ]:
plt.figure(figsize=(8,6))

rfm['Segment'].value_counts().plot(
    kind='bar',
    color='skyblue'
)

plt.title("Customer Segment Distribution")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    data=rfm,
    x='Segment',
    y='Monetary'
)

plt.xticks(rotation=45)

plt.title("Customer Spending by Segment")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=rfm,
    x='Recency',
    y='Frequency',
    hue='Segment',
    palette='Set2'
)

plt.title("Customer Behavior Segments")

plt.show()

In [ ]:
rfm.to_csv("../data/customer_segments.csv")

## Step 16: Business Insights

#### Champions
High-value customers with frequent purchases.

Strategy: Offer VIP programs and exclusive deals.

#### Loyal Customers
Regular customers with strong engagement.

Strategy: Provide loyalty rewards and personalized recommendations.

#### New Customers
Recently acquired customers.

Strategy: Send onboarding offers and product suggestions.

#### At Risk Customers
Customers who haven't purchased recently.

Strategy: Run re-engagement campaigns.

## Step 17: 3D Customer Segmentation Visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8,6))

ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    rfm['Recency'],
    rfm['Frequency'],
    rfm['Monetary'],
    c=rfm['Cluster']
)

ax.set_xlabel("Recency")
ax.set_ylabel("Frequency")
ax.set_zlabel("Monetary")

plt.title("3D Customer Segmentation")

plt.show()

## Step 18: Export Segmentation Results

In [ ]:
rfm.reset_index(inplace=True)

rfm.to_csv("../data/customer_segments.csv", index=False)